# 02 — MTFL 학습

[`01_preprocess.ipynb`](01_preprocess.ipynb)에서 만든 **`train.txt` / `test.txt`** 와 `videos/` 를 사용합니다.  
MTFL 원본 코드는 [`models/MTFL`](../models/MTFL/) — **수정 없이 subprocess만** 호출합니다.

## 전체 흐름

1. **Swin3D feature** — L8 / L32 / L64 (`feature_extractor.py`)
2. **개수 검증** — feature 파일 수 == `train.txt` 줄 수
3. **Detection 학습** — `detection/train.py`
4. **(선택) Test** — `detection/test.py` → `03_Result/results_eval/`
5. **(선택) 시각화** — `scripts/viz.py`

## 사전 준비

- `01` 완료: `{CUSTOM_ROOT}/annotations/train.txt` 존재
- ffmpeg (feature 추출용)
- OpenMMLab: mmaction2, mmcv, torch
- Swin 가중치: `02_Weights/pretrained/swin_base_....pth`


## 환경: Colab Drive 마운트


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass


## 경로·실행 플래그


In [ ]:
import sys
import subprocess
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/딥러닝 팀플")
WORKSPACE = DRIVE_ROOT / "04_Workspace"

sys.path.insert(0, str(WORKSPACE / "scripts"))
from config import (
    DATA_UCF, DATA_AIHUB, MTFL_ROOT, PRETRAINED_SWIN,
    CHECKPOINTS_DIR, RESULTS_EVAL,
)

DATASET = "ucf"  # ucf | aihub
CUSTOM_ROOT = DATA_UCF if DATASET == "ucf" else DATA_AIHUB / "MTFL_custom"
GPU = 0
RUN_FEATURES = True
RUN_TRAIN = True
RUN_TEST = True
RUN_VIZ = True

print("MTFL_ROOT:", MTFL_ROOT)
print("CUSTOM_ROOT:", CUSTOM_ROOT)
print("train.txt:", CUSTOM_ROOT / "annotations/train.txt")


## 사전 점검: ffmpeg · 패키지


In [ ]:
import shutil
print("ffmpeg:", shutil.which("ffmpeg"))
# Colab 예시:
# !pip install -q -r {WORKSPACE}/requirements.txt
# mmaction2 / mmcv 는 OpenMMLab 가이드에 따라 설치


## 1단계: Swin feature (L8 / L32 / L64)

| 스케일 | clip_length | 저장 경로 |
|--------|-------------|-----------|
| L8 | 8 | `{CUSTOM_ROOT}/features/L8` |
| L32 | 32 | `features/L32` |
| L64 | 64 | `features/L64` |

입력 영상: `{CUSTOM_ROOT}/videos`


In [ ]:
def run_feature_extraction(clip_length):
    save_dir = CUSTOM_ROOT / "features" / f"L{clip_length}"
    cmd = [
        sys.executable,
        str(MTFL_ROOT / "utils/feature_extractor.py"),
        "--clip_length", str(clip_length),
        "--dataset_path", str(CUSTOM_ROOT / "videos"),
        "--save_dir", str(save_dir),
        "--pretrained_3d", str(PRETRAINED_SWIN),
        "--batch_size", "4", "--num_workers", "0", "--gpu", str(GPU),
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=str(MTFL_ROOT), check=True)

if RUN_FEATURES:
    for cl in (8, 32, 64):
        print(f"\n========== L{cl} ==========")
        run_feature_extraction(cl)


### 1단계 확인: 스케일별 feature 개수


In [ ]:
train_lines = [
    ln for ln in (CUSTOM_ROOT / "annotations/train.txt").read_text(encoding="utf-8").splitlines()
    if ln.strip()
]
n_train = len(train_lines)
print("train.txt lines:", n_train)
for scale in ("L8", "L32", "L64"):
    n_feat = len(list((CUSTOM_ROOT / "features" / scale).rglob("*.txt")))
    print(f"  {scale}: {n_feat} feature files")


## 2단계: 개수 검증 (assert)

feature 추출이 끝난 뒤, **train.txt 한 줄당 feature 파일 하나**가 있어야 합니다.  
실패 시 `01_preprocess` 재실행 또는 feature 재추출을 확인하세요.


In [ ]:
if RUN_FEATURES:
    for scale in ("L8", "L32", "L64"):
        n_feat = len(list((CUSTOM_ROOT / "features" / scale).rglob("*.txt")))
        assert n_feat == n_train, f"{scale}: features={n_feat} != train.txt={n_train}"
    print("feature count OK:", n_train)


## 3단계: detection 학습

| 인자 | 경로 |
|------|------|
| `--train_anno` | `annotations/train.txt` |
| `--test_anno` | `annotations/test.txt` |
| `--save_models` | `02_Weights/checkpoints/` |


In [ ]:
def run_detection_train():
    cmd = [
        sys.executable, str(MTFL_ROOT / "detection/train.py"),
        "--model-name", "MTFL",
        "--train_anno", str(CUSTOM_ROOT / "annotations/train.txt"),
        "--test_anno", str(CUSTOM_ROOT / "annotations/test.txt"),
        "--lf_dir", str(CUSTOM_ROOT / "features/L64"),
        "--mf_dir", str(CUSTOM_ROOT / "features/L32"),
        "--sf_dir", str(CUSTOM_ROOT / "features/L8"),
        "--save_models", str(CHECKPOINTS_DIR),
        "--output_dir", str(CUSTOM_ROOT / "train_results"),
        "--feature_size", "1024", "--seg_num", "32",
        "--batch-size", "64", "--workers", "0", "--gpu", str(GPU),
    ]
    subprocess.run(cmd, cwd=str(MTFL_ROOT), check=True)

if RUN_TRAIN:
    CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
    run_detection_train()


### 3단계 확인: checkpoint · 학습 로그


In [ ]:
if RUN_TRAIN:
    ckpts = sorted(CHECKPOINTS_DIR.glob("MTFL-*.pkl"))
    print("checkpoints:", [p.name for p in ckpts[-5:]])
    log_dir = CUSTOM_ROOT / "train_results"
    if log_dir.exists():
        print("train log dir:", log_dir)
        print("  (step별 AUC/AP txt 파일 확인)")


## 4단계 (선택): test set 추론


In [ ]:
def run_detection_test(ckpt_name="MTFL-1280.pkl"):
    out = RESULTS_EVAL
    out.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, str(MTFL_ROOT / "detection/test.py"),
        "--test_anno", str(CUSTOM_ROOT / "annotations/test.txt"),
        "--detection_model", str(CHECKPOINTS_DIR / ckpt_name),
        "--lf_dir", str(CUSTOM_ROOT / "features/L64"),
        "--mf_dir", str(CUSTOM_ROOT / "features/L32"),
        "--sf_dir", str(CUSTOM_ROOT / "features/L8"),
        "--output_dir", str(out),
        "--feature_size", "1024", "--seg_num", "32", "--workers", "0", "--gpu", str(GPU),
    ]
    subprocess.run(cmd, cwd=str(MTFL_ROOT), check=True)

if RUN_TEST:
    run_detection_test()


## 5단계 (선택): 점수 시각화


In [ ]:
if RUN_VIZ:
    from viz import plot_saved_score
    score_dir = RESULTS_EVAL / "scores"
    if score_dir.exists():
        for p in sorted(score_dir.glob("*_scores.npy"))[:1]:
            plot_saved_score(p)
    else:
        print("no scores yet:", score_dir)
